In [1]:

import os, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"  
os.environ["OMP_NUM_THREADS"] = "2"             
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from src.config import (
    FAISS_INDEX_PATH, FAISS_IDMAP_PATH,
    CHUNKS_PARQUET, EMBED_MODEL, E5_QUERY_PREFIX
)


EMBED_MODEL = "intfloat/multilingual-e5-small"

model = SentenceTransformer(EMBED_MODEL)
index = faiss.read_index(FAISS_INDEX_PATH)
id_map = np.load(FAISS_IDMAP_PATH)
df = pd.read_parquet(CHUNKS_PARQUET)

print(" Model + FAISS index loaded successfully.")
print(f"Chunks available: {len(df)}")


d:\kannada-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Model + FAISS index loaded successfully.
Chunks available: 19


In [2]:
def search_kannada_query(query_text, top_k=3):
    q_emb = model.encode(
        [E5_QUERY_PREFIX + query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    scores, ids = index.search(q_emb, top_k)
    top_ids = id_map[ids[0]]

    rows = []
    for rank, (i, s) in enumerate(zip(top_ids, scores[0]), 1):
        r = df.iloc[int(i)]
        rows.append({
            "Rank": rank,
            "Score": round(float(s), 4),
            "Page": int(r["page"]),
            "Text": r["text"][:400].replace("\n", " ") + "..."
        })
    return pd.DataFrame(rows)


In [3]:
queries = [
    "ವಿಕಿಪೀಡಿಯ ಎಂದರೇನು?",
    "ಖಾತೆ ತೆರೆಯುವ ಕ್ರಮ ಯಾವುದು?",
    "ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆ ಹೇಗೆ ಮಾಡುವುದು?"
]

for q in queries:
    print(f"\n ಪ್ರಶ್ನೆ: {q}\n")
    display(search_kannada_query(q))
    print("="*80)



 ಪ್ರಶ್ನೆ: ವಿಕಿಪೀಡಿಯ ಎಂದರೇನು?



,Rank,Score,Page,Text
0,1,0.8850,2,ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆಯ ಟ್ಯುಟೆ ೀರಿಯಲ್ 1 ವಿಕಿಪ...
1,2,0.8718,4,ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆಯ ಟ್ಯುಟೆ ೀರಿಯಲ್ 3 ಕನ್ನಡ...
2,3,0.8671,11,ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆಯ ಟ್ಯುಟೆ ೀರಿಯಲ್ 10 ಸ ಚನ...



 ಪ್ರಶ್ನೆ: ಖಾತೆ ತೆರೆಯುವ ಕ್ರಮ ಯಾವುದು?



,Rank,Score,Page,Text
0,1,0.8388,4,ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆಯ ಟ್ಯುಟೆ ೀರಿಯಲ್ 3 ಕನ್ನಡ...
1,2,0.8296,4,ಂದಯ ಬರೆದ ಬಟ್ನ್ ಮೀಲೆ ಕಿಿಕ್ಸ ಮಾಡಿ. ಬಳಕೆದಾರ ಖಾತೆ ...
2,3,0.8278,5,ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆಯ ಟ್ಯುಟೆ ೀರಿಯಲ್ 4 ಗೆ ತಾ...



 ಪ್ರಶ್ನೆ: ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆ ಹೇಗೆ ಮಾಡುವುದು?



,Rank,Score,Page,Text
0,1,0.9070,4,ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆಯ ಟ್ಯುಟೆ ೀರಿಯಲ್ 3 ಕನ್ನಡ...
1,2,0.8946,7,ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆಯ ಟ್ಯುಟೆ ೀರಿಯಲ್ 6 ಕನ್ನಡ...
2,3,0.8907,8,ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆಯ ಟ್ಯುಟೆ ೀರಿಯಲ್ 7 ಸೆೀರಿ...


In [4]:
def show_full_chunks(query_text, top_k=3):
    """Display the full text of top-k retrieved Kannada chunks"""
    q_emb = model.encode(
        [E5_QUERY_PREFIX + query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    scores, ids = index.search(q_emb, top_k)
    top_ids = id_map[ids[0]]

    print(f"\n🔍 ಪ್ರಶ್ನೆ: {query_text}")
    print("="*90)
    for rank, (i, s) in enumerate(zip(top_ids, scores[0]), 1):
        r = df.iloc[int(i)]
        print(f"\n#️⃣ Rank {rank} |  Page {r['page']} |  Score {round(float(s),4)}")
        print("-"*90)
        print(r["text"])  
        print("\n" + "="*90)


show_full_chunks("ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆ ಹೇಗೆ ಮಾಡುವುದು?", top_k=3)



🔍 ಪ್ರಶ್ನೆ: ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆ ಹೇಗೆ ಮಾಡುವುದು?

#️⃣ Rank 1 |  Page 4 |  Score 0.907
------------------------------------------------------------------------------------------
ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆಯ ಟ್ಯುಟೆ ೀರಿಯಲ್ 3 ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದಕರಾಗಯವುದಯ ವಿಕಿಪೀಡಿಯಕೆೆ ಲೆೀಖನ್ ಸೆೀರಿಸಬೆೀಕಾದರೆ ನೀವು ಮೊದಲ್ಯ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದಕರಾಗಿ ನೆ ೀಂದಾಯಿತ್ರಾಗಿರಬೆೀಕಯ. ಇದಕಾೆಗಿ ನೀವು ಕನ್ನಡ ವಿಕಿಪೀಡಿಯದಲ್ಲಿ ಖಾತೆ ಹೆ ಂದಿರಬೆೀಕಯ. ಇದಯ ಬಹಳ ಸಯಲ್ಭದ ಕೆಲ್ಸ. ಮೊತ್ುಮೊದಲ್ಯ ಕನ್ನಡ ವಿಕಿಪೀಡಿಯವನ್ಯನ ತೆರೆಯಿರಿ. ನ್ಂತ್ರ ಮೀಲ್ೆಡೆ ಬಲ್ಭಾಗದಲ್ಲಿ “ಹೆ ಸ ಖಾತೆ ತೆರೆಯಿರಿ” ಎಂದಯ ಬರೆದ ಅಕ್ಷರಗಳ ಮೀಲೆ ಕಿಿಕ್ಸ ಮಾಡಿ. ಈಗ ಒಂದಯ ಪ್ಯಟ್ ತೆರೆದಯಕೆ ಳಯೆತ್ುದೆ. ಬಳಕೆದಾರ ಹೆಸರಯ ಎಂಬಲ್ಲಿ ವಿಕಿಪೀಡಿಯದಲ್ಲಿ ನಮಮ ಹೆಸರಯ ಹೆೀಗಿರಬೆೀಕಯ ಎಂಬಯದನ್ಯನ ನ್ಮ ದಿಸಿ. ನಮಮ ಹೆಸರಯ ತ್ಯಂಬ ಸಾಮಾನ್ು ಹೆಸರಾದರೆ ಅದನ್ಯನ ಈಗಾಗಲೆ ಯಾರಾದರ ತೆಗೆದಯಕೆ ಂಡಿರಯವ ಸಾಧ್ಯತೆ ಇರಯತ್ುದೆ. ಆದಯದರಿಂದ ನಮಮ ಪ್ ತ್ತಯ ಹೆಸರನ್ಯನ ಬಳಸಯವುದಯ ಒಳ್ೆೆಯದಯ. ಪ್ರವೆೀಶ್ಪ್ದ ಎಂಬಲ್ಲಿ ನಮಗೆ ಯಾವ ಪಾಸ ವರ್ಡಯ ಬೆೀಕೆ ೀ ಅದನ್ಯನ ಟೆೈಪ್ ಮಾಡಿ. ಪ್ರವೆೀಶ್ಪ್ದವನ್ಯನ ದೃಢೀಕರಿಸಿ ಎಂಬಲ್ಲಿ ಅದನೆನೀ ಮತೆ ುಮಮ ಟೆೈಪ್ ಮಾಡಿ. ಮಂಚಂಚೆ ವಿಳ್ಾಸ ಎಂಬಲ್ಲಿ ನಮಮ ಇಮೈಲ್ ವಿಳ್ಾಸವನ್ಯನ ನೀಡಿ. ಸಯರಕೆೆ ನಗರಹಿಸಯ ಎಂಬಲ್ಲಿ ಅಲೆಿೀ 

##Generate Kannada Answer from Retrieved Chunks

In [5]:
from importlib import reload
import src.generate_answer_groq as gag
reload(gag)
from src.generate_answer_groq import generate_kannada_answer


In [10]:

question = "ವಿಕಿಪೀಡಿಯ ಎಂದರೇನು?"   
q_emb = model.encode([E5_QUERY_PREFIX + question],
                     convert_to_numpy=True,
                     normalize_embeddings=True)

# Retrieve top-3 matching chunks
scores, ids = index.search(q_emb, 3)
best_score = float(scores[0][0])  # top similarity score
context = "\n\n".join(df.iloc[int(i)]["text"] for i in ids[0])

# --- Threshold: if similarity too low, skip answer generation ---
THRESHOLD = 0.82  # tune between 0.80–0.85

print(f"Best similarity score: {best_score:.4f}\n")

if best_score < THRESHOLD:
    print(" ನೀಡಲಾದ ದಾಖಲೆಗಳಲ್ಲಿ ಈ ಪ್ರಶ್ನೆಗೆ ಸಂಬಂಧಿಸಿದ ಮಾಹಿತಿ ಇಲ್ಲ.\n")
else:
    answer = generate_kannada_answer(question, context)
    print("💡 ಪೂರ್ಣ ಉತ್ತರ:\n", answer)
    await speak_kannada(answer, voice="kn-IN-SapnaNeural")


Best similarity score: 0.8850

💡 ಪೂರ್ಣ ಉತ್ತರ:
 ವಿಕಿಪೀಡಿಯ ಎಂದರೇನು? ವಿಕಿಪೀಡಿಯ ಎಂಬುದು ಒಂದು ಮಾಹಿತಿ ಸಂಗ್ರಹಣೆಯ ಮತ್ತು ವಿಕಿಪೀಡಿಯ ಒಂದಯ ಸವತ್ಂತ್ರ ಮತ್ತು ಮಯಕು ವಿಶ್ವಕೆ ೀಶ್. ಇದಯ ೨೦೦೧ರಲ್ಲಿ ಪಾರರಂಭವಾಯಿತ್ಯ. ಇದಯ ಜಗತ್ತುನ್ ೨೮೬ ಭಾಷೆಗಳಲ್ಲಿದೆ. ಇನ್ ನ ಸಯಮಾರಯ ೩೦೦ ಭಾಷೆಯ ವಿಕಿಪೀಡಿಯಗಳಯ ತ್ಯಾರಿಯ ಹಂತ್ದಲ್ಲಿವೆ. ಸಯಮಾರಯ ೨೦ ಚಿಲ್ಿರೆ ಭಾರತ್ತೀಯ ಭಾಷೆಗಳಲ್ಲಿ ವಿಕಿಪೀಡಿಯ ಲ್ಭಯ. ಅದರಲ್ಲಿ ಕನ್ನಡವೂ ಸೆೀರಿದೆ. ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ೨೦೦೩ರಲ್ಲಿ ಪಾರರಂಭವಾಯಿತ್ಯ.


🔊 ಕನ್ನಡ ಧ್ವನಿ ಪ್ಲೇ ಆಗುತ್ತಿದೆ...


In [9]:
import asyncio, edge_tts
from IPython.display import Audio, display

# ✅ Voice Function (Edge Neural Kannada Voice)
async def speak_kannada(text, voice="kn-IN-SapnaNeural", file_path="kannada_answer.mp3"):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(file_path)
    display(Audio(file_path, autoplay=True))
    print("🔊 ಕನ್ನಡ ಧ್ವನಿ ಪ್ಲೇ ಆಗುತ್ತಿದೆ...")


question = "ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆ ಹೇಗೆ ಮಾಡುವುದು?" 


q_emb = model.encode([E5_QUERY_PREFIX + question],
                     convert_to_numpy=True,
                     normalize_embeddings=True)

scores, ids = index.search(q_emb, 3)
best_score = float(scores[0][0])  
context = "\n\n".join(df.iloc[int(i)]["text"] for i in ids[0])

THRESHOLD = 0.82  

print(f"🔎 Best similarity score: {best_score:.4f}\n")


if best_score < THRESHOLD:
    print(" ನೀಡಲಾದ ದಾಖಲೆಗಳಲ್ಲಿ ಈ ಪ್ರಶ್ನೆಗೆ ಸಂಬಂಧಿಸಿದ ಮಾಹಿತಿ ಇಲ್ಲ.\n")
else:
    answer = generate_kannada_answer(question, context)
    print(" ಪೂರ್ಣ ಉತ್ತರ:\n", answer)

    await speak_kannada(answer, voice="kn-IN-SapnaNeural")


🔎 Best similarity score: 0.9070

 ಪೂರ್ಣ ಉತ್ತರ:
 ಕನ್ನಡ ವಿಕಿಪೀಡಿಯ ಸಂಪಾದನೆ ಮಾಡಲು ನೀವು ಮೊದಲಿಗೆ ಕನ್ನಡ ವಿಕಿಪೀಡಿಯದಲ್ಲಿ ಖಾತೆ ಹೊಂದಬೇಕಾಗುತ್ತದೆ. ಖಾತೆ ಹೊಂದಲು ನೀವು ಕನ್ನಡ ವಿಕಿಪೀಡಿಯದ ಮುಖ್ಯ ಪುಟದಲ್ಲಿ "ಹೆಸರಿನ ಖಾತೆ ತೆರೆಯಿರಿ" ಎಂದು ಬರೆದ ಅಕ್ಷರಗಳ ಮೇಲೆ ಕ್ಲಿಕ್ ಮಾಡಬೇಕಾಗುತ್ತದೆ. ನಂತರ ಒಂದು ಪೇಟ್ ತೆರೆದಾಗ, ನೀವು ಬಳಕೆದಾರನ ಹೆಸರನ್ನು ನೀಡಬೇಕಾಗುತ್ತದೆ. ನೀವು ನೀಡಿದ ಹೆಸರು ಇತರ ಬಳಕೆದಾರರಿಂದ ತೆಗೆದುಕೊಳ್ಳದಿರಲು ನೀವು ಒಂದು ಸಾಮಾನ್ಯ ಹೆಸರನ್ನು ಬಳಸಬೇಕಾಗುತ್ತದೆ.

ನಂತರ, ನೀವು ಪ್ರವೇಶಿಸುವ ಪದದಲ್ಲಿ ನೀವು ಪಾಸ್‌ವರ್ಡ್ ನೀಡಬೇಕಾಗುತ್ತದೆ. ನೀವು ಪಾಸ್‌ವರ್ಡ್ ನೀಡಿದ ನಂತರ, ನೀವು ನಮಗೆ ಸಂಬಂಧಿಸಿದ ಇ-ಮೇಲ್ ವಿಳಾಸವನ್ನು ನೀಡಬೇಕಾಗುತ್ತದೆ. ನಂತರ, ನೀವು ನಮಗೆ ಸಂಬಂಧಿಸಿದ ಅಲೆಕ್ಸಾ ವಿಳಾಸವನ್ನು ನೀಡಬೇಕಾಗುತ್ತದೆ.

ನಂತರ, ನೀವು "ಖಾತೆಯನ್ನು ಸೃಷ್ಟಿಸಿ" ಎಂದು ಬರೆದ ಕ್ಲಿಕ್ ಮಾಡಬೇಕಾಗುತ್ತದೆ. ನಂತರ, ನೀವು ನಮಗೆ ಸಂಬಂಧಿಸಿದ ವಿವರಗಳನ್ನು ನೀಡಬೇಕಾಗುತ್ತದೆ.

ನಂತರ, ನೀವು ಕನ್ನಡ ವಿಕಿಪೀಡಿಯದಲ್ಲಿ ಲೆಖನವನ್ನು ಸೃಷ್ಟಿಸಬೇಕಾಗುತ್ತದೆ. ಲೆಖನವನ


🔊 ಕನ್ನಡ ಧ್ವನಿ ಪ್ಲೇ ಆಗುತ್ತಿದೆ...
